#### nb_MigrateSemanticModels
The Purpose of this notebook is to scan a tenant for workspaces containing Large Format semantic models and migrate the models to a new workspace.  This is specifically to facilitate cross-region migrations.  Note the script will migrate Small Format semantic models as well if they are co-located in the same workspace as a Large Format model. 


In [ ]:
import sempy_labs as labs
import sempy_labs.admin as admin
import sempy.fabric as fabric
import pandas as pd
import json
import time
from datetime import datetime
from typing import Dict, Iterable, List, Optional, Tuple

In [ ]:
# Params

DEBUG = False

# The target capacity id you are migrating to
CAPACITY_ID = "YourTargetCapacityIdHere" 

# Maximum number of partitions to refresh concurrently per model
MAX_CONCURRENT_PARTITIONS = 3

# Seconds to wait between polling refresh status
REFRESH_POLL_INTERVAL_SECONDS = 15

In [ ]:
def list_workspace_ids() -> List[str]:
    mods = admin.list_workspaces(exclude_personal_workspaces=True, exclude_inactive_workspaces=True)
    
    if DEBUG:
        print(mods['Id'])

    return mods['Id']

In [ ]:
def scan_workspaces(workspace_ids: List[str]) -> List[dict]:
    scan = admin.scan_workspaces(workspace=workspace_ids)
            
    return scan

In [ ]:
def iter_items_from_workspace(ws: dict) -> Iterable[Tuple[str, dict]]:
    # Yields (item_type, item_dict) across ALL arrays under the workspace block.
    # Treat any top-level array of dicts that looks like artifacts (has an 'id') as items.
    # Ignore these arrays which are not top-level artifacts
    non_artifact_keys = {"users", "tags", "dashboards.tiles"}  # extend as needed

    for key, val in ws.items():
        if key in {"id", "name", "type", "state", "capacityId", "isOnDedicatedCapacity", "defaultDatasetStorageFormat"}:
            continue
        if key in non_artifact_keys:
            continue
        if isinstance(val, list) and val and isinstance(val[0], dict) and "id" in val[0]:
            yield (key, val)

In [ ]:
def flatten_scan_results(scan_results: List[dict]) -> pd.DataFrame:
    #Converts scan JSON => pandas DataFrame with one row per artifact.
    records: List[dict] = []

    for ws in (scan_results or {}).get("workspaces", []):
        ws_id = ws.get("id")
        ws_name = ws.get("name")
        for item_type, items in iter_items_from_workspace(ws):
            for it in items:
                rec = {
                    "workspace_id": ws_id,
                    "workspace_name": ws_name,
                    "item_type": item_type,                     # e.g., 'reports', 'dashboards', 'datasets', 'lakehouses', ...
                    "item_id": it.get("id"),
                    "item_name": it.get("name") or it.get("displayName"),
                    "targetStorageMode": it.get("targetStorageMode"),
                    "modifiedBy": it.get("modifiedBy")
                }
                records.append(rec)

    df = pd.DataFrame.from_records(records)

    return df

In [ ]:
def create_new_workspace(capacity_id: str, workspace_name: str, fabric_client: fabric.FabricRestClient, pbi_client: fabric.PowerBIRestClient):
    #Create a new workspace with the name_new and assign the capacity id
    new_workspace_name = workspace_name + '_new'

    payload = {
        "displayName": new_workspace_name,
        "description": "Workspace created via SemPy FabricRestClient",
        "capacityId": capacity_id
    }

    resp = fabric_client.post("/v1/workspaces", json=payload)
    new_ws_id = resp.json().get("id")
    
    #Update the default storage format
    payload2 = {
        "defaultDatasetStorageFormat": "Large"
    }
    
    pbi_client.patch(f"https://api.powerbi.com/v1.0/myorg/groups/{new_ws_id}", json=payload2)

    return resp

In [ ]:
# ----------------------------
# Orchestration
# ----------------------------

fabric_client = fabric.FabricRestClient()
pbi_client = fabric.PowerBIRestClient()

ws_ids = list_workspace_ids()

if DEBUG:
    print(f"Workspaces Found: {len(ws_ids)}")

scans = scan_workspaces(ws_ids)

# Flatten to artifacts
df_all = flatten_scan_results(scans)

if DEBUG:
    print(f"Artifacts discovered across modified workspaces: {len(df_all)}")


# df_all.sort_values(["workspace_name","item_type","item_name"], inplace=True, ignore_index=True)

ws_with_premiumfiles = df_all.loc[df_all["targetStorageMode"] == "PremiumFiles", "workspace_id"].unique()

df_datasets_to_migrate = df_all[(df_all["workspace_id"].isin(ws_with_premiumfiles)) & (df_all["item_type"] == "datasets") & (df_all["item_name"] != "Report Usage Metrics Model")]

if DEBUG:
    df_datasets_to_migrate

#get a distinct list of workspaces we need to migrate
distinct_workspaces = list(
    df_datasets_to_migrate[["workspace_id", "workspace_name"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

current_to_new = {}

#create new workspaces and the mapping of new to old ids
for current_ws_id, current_ws_name in distinct_workspaces:
        resp = create_new_workspace(CAPACITY_ID, current_ws_name, fabric_client, pbi_client )
        new_ws_id = resp.json().get("id")
        current_to_new[current_ws_id] = new_ws_id

if DEBUG:
    print(current_to_new)

df_datasets_to_migrate["new_workspace_id"] = df_datasets_to_migrate["workspace_id"].map(current_to_new)

#deploy the model metadata to the new workspaces
for _, row in df_datasets_to_migrate.iterrows():
        labs.deploy_semantic_model(source_dataset=row['item_name'], source_workspace=row['workspace_id'], target_dataset=row['item_name'], target_workspace=row['new_workspace_id'], refresh_target_dataset=False)

In [ ]:
#TODO: refresh partitions

